<span style="color:olive; font-size:40px">**DATA EXTRACTION**</span>

<span style="font-size:16px; font-weight:bold"> In this notebook, data will be extracted from the original source files and processed to generate a new dataset containing the variables required for subsequent analysis. </span> 

<span style="font-size:20px; color:red; font-weight:bold"> <strong>Note:</strong> All newly created notebooks must include an initial cell for environment setup and memory usage management. </span>

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import psutil, gc

sc.settings.n_jobs = 4
sc.settings.verbosity = 2
sc.settings.figdir = 'scrna_gbm/results/'
np.random.seed(42)

def check_ram():
    ram = psutil.Process().memory_info().rss / 1e9
    print(f"RAM en uso: {ram:.1f} GB")
    if ram > 11:
        print("⚠️ RAM alta — considera reiniciar el kernel pronto")

<span style="font-size:20px; color:olive; font-weight:bold">Data extraction from ZIP files</span> <br><br>
<span style="font-size:14px">***The original preprocessed data is stored in ZIP files. The following code will extract the necessary files from these ZIP archives and save them in a specified directory for further processing.***</span>

<span style="font-size:14px; color:red">**Since the analysis is conducted in a Windows-based environment, files associated with macOS (i.e., the <code>__MACOSX</code> directory) are not required and will be removed during extraction.**</span>

<span style="font-size:14px">**Our directory structure is nested; therefore, the extraction process must be carried out in two stages. First, the archive containing the patient folders is extracted, followed by the extraction of the individual files within each patient folder.**</span>

In [25]:
import zipfile
from pathlib import Path

main_zip = Path(r"C:\Users\dioul\OneDrive\Documentos\scRNA_raw_matrix.zip")
output_folder = Path(r"C:\Users\dioul\OneDrive\Documentos\scRNA-seq_Analysis\scrna_gbm\data")

print("Re-extracting all patients cleanly...\n")

with zipfile.ZipFile(main_zip, 'r') as z:
    for member in z.namelist():
        # Skip macOS garbage files
        if '__MACOSX' in member or '.DS_Store' in member or member.startswith('._'):
            continue
        z.extract(member, output_folder)
        
print("Extraction done! Verifying all patients...\n")

raw_dir = output_folder / 'Raw_matrix'
for patient_dir in sorted(raw_dir.iterdir()):
    if not patient_dir.is_dir():
        continue
    # Find the matrix folder (handles Pt19's different subfolder name)
    matrix_folders = list(patient_dir.rglob('*.mtx*'))
    if matrix_folders:
        print(f"  ✅ {patient_dir.name}: {len(matrix_folders)} matrix files found")
    else:
        print(f"  ❌ {patient_dir.name}: No matrix files found")

Re-extracting all patients cleanly...

Extraction done! Verifying all patients...

  ✅ Pt1: 1 matrix files found
  ✅ Pt10: 1 matrix files found
  ✅ Pt11: 1 matrix files found
  ✅ Pt12: 1 matrix files found
  ✅ Pt13: 1 matrix files found
  ✅ Pt14: 1 matrix files found
  ✅ Pt15: 1 matrix files found
  ✅ Pt16: 1 matrix files found
  ✅ Pt17: 1 matrix files found
  ✅ Pt18: 1 matrix files found
  ✅ Pt19: 1 matrix files found
  ✅ Pt2: 1 matrix files found
  ✅ Pt20: 1 matrix files found
  ✅ Pt21: 1 matrix files found
  ✅ Pt22: 1 matrix files found
  ✅ Pt23: 1 matrix files found
  ✅ Pt24: 1 matrix files found
  ✅ Pt3: 1 matrix files found
  ✅ Pt4: 1 matrix files found
  ✅ Pt5: 1 matrix files found
  ✅ Pt6: 1 matrix files found
  ✅ Pt7: 1 matrix files found
  ✅ Pt8: 1 matrix files found
  ✅ Pt9: 1 matrix files found


<span style="font-size:16px; color: olive">**Mapping what subfolder each patient uses:**</span>

In [28]:
raw_dir = Path(r"C:\Users\dioul\OneDrive\Documentos\scRNA-seq_Analysis\scrna_gbm\data\Raw_matrix")

print("Subfolder structure per patient:\n")
for patient_dir in sorted(raw_dir.iterdir(), key=lambda x: int(''.join(filter(str.isdigit, x.name)))):
    if not patient_dir.is_dir():
        continue
    subfolders = [f.name for f in patient_dir.iterdir() if f.is_dir()]
    print(f"  {patient_dir.name}: {subfolders}")

Subfolder structure per patient:

  Pt1: ['filtered_feature_bc_matrix']
  Pt2: ['filtered_feature_bc_matrix']
  Pt3: ['filtered_feature_bc_matrix']
  Pt4: ['filtered_feature_bc_matrix']
  Pt5: ['filtered_feature_bc_matrix']
  Pt6: ['filtered_feature_bc_matrix']
  Pt7: ['filtered_feature_bc_matrix']
  Pt8: ['filtered_feature_bc_matrix']
  Pt9: ['filtered_feature_bc_matrix']
  Pt10: ['filtered_feature_bc_matrix']
  Pt11: ['filtered_feature_bc_matrix']
  Pt12: ['filtered_feature_bc_matrix']
  Pt13: ['filtered_feature_bc_matrix']
  Pt14: ['filtered_feature_bc_matrix']
  Pt15: ['filtered_feature_bc_matrix']
  Pt16: ['filtered_feature_bc_matrix']
  Pt17: ['filtered_feature_bc_matrix']
  Pt18: ['filtered_feature_bc_matrix']
  Pt19: ['Pt19_Matrix']
  Pt20: ['Pt20_matrix']
  Pt21: ['Pt21_matrix']
  Pt22: ['Pt22_matrix']
  Pt23: ['Pt23_matrix']
  Pt24: ['Pt24_matrix']


In [29]:
import scanpy as sc
from pathlib import Path
import gc

raw_dir = Path(r"C:\Users\dioul\OneDrive\Documentos\scRNA-seq_Analysis\scrna_gbm\data\Raw_matrix")

def find_matrix_folder(patient_dir):
    """Finds the matrix subfolder regardless of its name."""
    for subfolder in patient_dir.iterdir():
        if subfolder.is_dir() and any(subfolder.rglob('*.mtx*')):
            return subfolder
    return None

print("Loading all 24 patients...\n")
errors = []

for patient_dir in sorted(raw_dir.iterdir(), key=lambda x: int(''.join(filter(str.isdigit, x.name)))):
    if not patient_dir.is_dir():
        continue

    fbm = find_matrix_folder(patient_dir)
    if fbm is None:
        print(f"  ❌ {patient_dir.name}: matrix folder not found")
        errors.append(patient_dir.name)
        continue

    try:
        ad = sc.read_10x_mtx(str(fbm), var_names='gene_symbols', gex_only=True)
        print(f"  ✅ {patient_dir.name}: {ad.n_obs:,} cells | {ad.n_vars:,} genes | folder: {fbm.name}")
        del ad; gc.collect()
    except Exception as e:
        print(f"  ❌ {patient_dir.name}: {e}")
        errors.append(patient_dir.name)

print(f"\n{'✅ All 24 patients loaded successfully!' if not errors else f'❌ Errors in: {errors}'}")

Loading all 24 patients...

  ✅ Pt1: 2,896 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt2: 670 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt3: 1,576 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt4: 2,524 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt5: 7,315 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt6: 2,851 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt7: 1,566 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt8: 7,594 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt9: 12,617 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt10: 2,606 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt11: 3,753 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt12: 1,894 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt13: 3,064 cells | 33,694 genes | folder: filtered_feature_bc_matrix
  ✅ Pt14: 2,857 cell

<span style="color: olive; font-size:16px">**Assessment of Key Genes Presence**</span>

<span style="font-size:14px">We will verify the presence of key genes in the dataset, such as <code>EGFR</code>, <code>PDGFRA</code>, and <code>PTEN</code>, which are commonly associated with glioblastoma. This step ensures that our dataset contains relevant information for subsequent analyses.</span>

<span style="font-size:14px">We will check the Pt1 patient folder for the presence of the key genes. If these genes are present, we can proceed with confidence that our dataset is suitable for further analysis.</span>

In [30]:
test_path = r"C:\Users\dioul\OneDrive\Documentos\scRNA-seq_Analysis\scrna_gbm\data\Raw_matrix\Pt1\filtered_feature_bc_matrix"
adata_test = sc.read_10x_mtx(test_path, var_names='gene_symbols', gex_only=True)
print(adata_test)

key_genes = [
    # Angiogénesis
    'VEGFA','VEGFB','KDR','ESM1','DLL4','ANGPT2',
    # T γδ
    'TRDC','TRGC1','TRGC2','TRDV1','TRDV2','TRGV9',
    # General markers for microglia/macrophages
    'CD3D','CD8A','CD68','TMEM119','PECAM1','SIGLEC9'
]
print("\nKey genes:")
for g in key_genes:
    state = "✓" if g in adata_test.var_names else "✗ MISSING"
    print(f"  {state}  {g}")

del adata_test; gc.collect()    # Why? To free up RAM immediately after this test, since we only needed it to check gene presence.

AnnData object with n_obs × n_vars = 2896 × 33694
    var: 'gene_ids', 'feature_types'

Key genes:
  ✓  VEGFA
  ✓  VEGFB
  ✓  KDR
  ✓  ESM1
  ✓  DLL4
  ✓  ANGPT2
  ✓  TRDC
  ✓  TRGC1
  ✓  TRGC2
  ✓  TRDV1
  ✓  TRDV2
  ✓  TRGV9
  ✓  CD3D
  ✓  CD8A
  ✓  CD68
  ✓  TMEM119
  ✓  PECAM1
  ✓  SIGLEC9


139

<span style="color:olive; font-size:16px">**Data integrity ckeck**</span>

In [31]:
raw_dir = Path(r"C:\Users\dioul\OneDrive\Documentos\scRNA-seq_Analysis\scrna_gbm\data\Raw_matrix")

def find_matrix_folder(patient_dir):
    for subfolder in patient_dir.iterdir():
        if subfolder.is_dir() and any(subfolder.rglob('*.mtx*')):
            return subfolder
    return None

# Clinical group from the paper
grupo_map = {
    'Pt1':'ND',  'Pt2':'ND',  'Pt3':'ND',  'Pt4':'ND',  'Pt5':'ND',
    'Pt6':'Rec', 'Pt7':'Rec', 'Pt8':'Rec', 'Pt9':'Rec', 'Pt10':'Rec',
    'Pt11':'Non-res','Pt12':'Non-res','Pt13':'Non-res',
    'Pt14':'Non-res','Pt15':'Non-res',
    'Pt16':'Res','Pt17':'Res','Pt18':'Res','Pt19':'Res',
    'Pt20':'Res','Pt21':'Res','Pt22':'Res','Pt23':'Res','Pt24':'Res',
}

print("=" * 60)             #*60 characters of "=" for a clear visual separation
print("DATA INTEGRITY CHECK — 24 GBM patients")
print("=" * 60)

total_cells = 0
total_patients = 0
issues = []

for patient_dir in sorted(raw_dir.iterdir(), key=lambda x: int(''.join(filter(str.isdigit, x.name)))):
    if not patient_dir.is_dir():
        continue

    pid = patient_dir.name
    fbm = find_matrix_folder(patient_dir)
    if fbm is None:
        issues.append(f"{pid}: matrix folder not found")
        continue

    try:
        ad = sc.read_10x_mtx(str(fbm), var_names='gene_symbols', gex_only=True)

        # --- Checks ---
        flags = []

        # 1. Minimum viable cell count
        if ad.n_obs < 100:                    #Why 100? Because datasets with fewer than 100 cells are often too sparse for meaningful analysis.
            flags.append(f"⚠️ very few cells: {ad.n_obs}")

        # 2. No empty barcodes          #why? Datasets with 0 cells indicate a loading issue or a completely empty sample, which is a critical problem. 
        if ad.n_obs == 0:
            flags.append("❌ EMPTY — no cells")

        # 3. Key genes present Already checked in the test, but we can do it here for each patient to be sure every patient has them.
        key_genes = ['VEGFA', 'KDR', 'TRDC', 'CD3D', 'CD68', 'PECAM1']
        missing = [g for g in key_genes if g not in ad.var_names]
        if missing:
            flags.append(f"⚠️ missing key genes: {missing}")

        # 4. Clinical group assigned
        condition = grupo_map.get(pid, 'UNKNOWN')
        if condition == 'UNKNOWN':
            flags.append("⚠️ no clinical group assigned")

        status = "✅" if not flags else "⚠️"
        flag_str = " | ".join(flags) if flags else "all checks passed"
        print(f"  {status} {pid:5s} | {condition:7s} | {ad.n_obs:6,} cells | {ad.n_vars:,} genes | {flag_str}")

        total_cells += ad.n_obs
        total_patients += 1
        del ad; gc.collect()

    except Exception as e:
        issues.append(f"{pid}: {e}")
        print(f"  ❌ {pid}: LOAD ERROR — {e}")

print("\n" + "=" * 60)
print(f"SUMMARY")
print(f"  Patients loaded:  {total_patients}/24")
print(f"  Total cells:      {total_cells:,}")
print(f"  Avg cells/patient:{total_cells//total_patients if total_patients else 0:,}")
if issues:
    print(f"\n  ❌ Issues found:")
    for issue in issues:
        print(f"     {issue}")
else:
    print(f"\n  ✅ No issues found — ready for Bloque 2 QC")
print("=" * 60)

DATA INTEGRITY CHECK — 24 GBM patients
  ✅ Pt1   | ND      |  2,896 cells | 33,694 genes | all checks passed
  ✅ Pt2   | ND      |    670 cells | 33,694 genes | all checks passed
  ✅ Pt3   | ND      |  1,576 cells | 33,694 genes | all checks passed
  ✅ Pt4   | ND      |  2,524 cells | 33,694 genes | all checks passed
  ✅ Pt5   | ND      |  7,315 cells | 33,694 genes | all checks passed
  ✅ Pt6   | Rec     |  2,851 cells | 33,694 genes | all checks passed
  ✅ Pt7   | Rec     |  1,566 cells | 33,694 genes | all checks passed
  ✅ Pt8   | Rec     |  7,594 cells | 33,694 genes | all checks passed
  ✅ Pt9   | Rec     | 12,617 cells | 33,694 genes | all checks passed
  ✅ Pt10  | Rec     |  2,606 cells | 33,694 genes | all checks passed
  ✅ Pt11  | Non-res |  3,753 cells | 33,694 genes | all checks passed
  ✅ Pt12  | Non-res |  1,894 cells | 33,694 genes | all checks passed
  ✅ Pt13  | Non-res |  3,064 cells | 33,694 genes | all checks passed
  ✅ Pt14  | Non-res |  2,857 cells | 33,694 genes |